# Classification: Static Features

Session-level means of all eye-tracking metrics are used as features to predict depression severity. Each metric is averaged across all stimulus scenes in a session, producing one value per session. After session-level aggregation, features are further averaged per user to reduce within-subject measurement noise.

Three tasks are tested:
- binary classification (depressed vs not)
- multi-class (severity groups; PHQ-9: 5 classes, BDI-II: 4 classes)
- regression (predict score directly)

Every task is run on two depression scales:
- **PHQ-9** (Patient Health Questionnaire, 9 items, range 0–27)
- **BDI-II** (Beck Depression Inventory, 21 items, range 0–63)

Running both scales provides a convergent-validity check: if the same gaze features predict both questionnaires similarly, the signal reflects the depression construct rather than questionnaire-specific content.

Three models are compared:
- Logistic / Ridge Regression
- Random Forest
- XGBoost

Across three feature sets:
- all features
- statistically significant
- theory-driven

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

from src.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance,
)

## 1. Build session-level dataset

### 1.1 Build session features

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
forms = spark.table("anima.forms")

stimulus_scenes = scene_metrics.filter(F.col("scene_type") == "stimulus")

session_metrics = stimulus_scenes.groupBy("session_id").agg(
    F.mean("fixation_count").alias("avg_fixation_count"),
    F.mean("mean_fixation_duration_ms").alias("avg_fixation_duration_ms"),
    F.mean("total_fixation_duration_ms").alias("avg_total_fixation_duration_ms"),
    F.mean("fixation_rate_per_sec").alias("avg_fixation_rate"),
    F.mean("fixation_bias").alias("avg_fixation_bias"),

    F.mean("scanpath_length").alias("avg_scanpath_length"),
    F.mean("saccade_count").alias("avg_saccade_count"),
    F.mean("saccade_rate_per_sec").alias("avg_saccade_rate"),
    F.mean("mean_saccade_amplitude").alias("avg_saccade_amplitude"),

    F.mean("blink_count").alias("avg_blink_count"),
    F.mean("blink_rate_per_min").alias("avg_blink_rate"),

    F.mean("transition_matrix_density").alias("avg_transition_density"),
    F.mean("gaze_transition_entropy").alias("avg_gaze_entropy"),

    F.mean("first_fixation_duration_ms").alias("avg_first_fixation_duration_ms"),
    F.mean("second_fixation_duration_ms").alias("avg_second_fixation_duration_ms"),

    F.avg(F.when(F.col("first_fixation_valence") == "negative", 1).otherwise(0)).alias("first_fix_prob_negative"),
    F.avg(F.when(F.col("first_fixation_valence") == "positive", 1).otherwise(0)).alias("first_fix_prob_positive"),
    F.avg(F.when(F.col("first_fixation_valence") == "neutral", 1).otherwise(0)).alias("first_fix_prob_neutral"),
    F.avg(F.when(F.col("second_fixation_valence") == "negative", 1).otherwise(0)).alias("second_fix_prob_negative"),
    F.avg(F.when(F.col("second_fixation_valence") == "positive", 1).otherwise(0)).alias("second_fix_prob_positive"),
    F.avg(F.when(F.col("second_fixation_valence") == "neutral", 1).otherwise(0)).alias("second_fix_prob_neutral"),

    F.mean("dwell_time_ms_negative").alias("avg_dwell_time_negative"),
    F.mean("dwell_time_ms_positive").alias("avg_dwell_time_positive"),
    F.mean("dwell_time_ms_neutral").alias("avg_dwell_time_neutral"),
    F.mean("dwell_time_500ms_negative").alias("avg_dwell_500ms_negative"),
    F.mean("dwell_time_500ms_positive").alias("avg_dwell_500ms_positive"),
    F.mean("dwell_time_500ms_neutral").alias("avg_dwell_500ms_neutral"),

    F.mean("fixation_proportion_negative").alias("avg_fix_proportion_negative"),
    F.mean("fixation_proportion_positive").alias("avg_fix_proportion_positive"),
    F.mean("fixation_proportion_neutral").alias("avg_fix_proportion_neutral"),

    F.mean("fixation_count_negative").alias("avg_fix_count_negative"),
    F.mean("fixation_count_positive").alias("avg_fix_count_positive"),
    F.mean("fixation_count_neutral").alias("avg_fix_count_neutral"),

    F.mean("revisit_count_negative").alias("avg_revisit_count_negative"),
    F.mean("revisit_count_positive").alias("avg_revisit_count_positive"),
    F.mean("revisit_count_neutral").alias("avg_revisit_count_neutral"),

    F.mean("ttff_ms_negative").alias("avg_ttff_negative"),
    F.mean("ttff_ms_positive").alias("avg_ttff_positive"),
    F.mean("ttff_ms_neutral").alias("avg_ttff_neutral"),
)

# Join with forms
df_joined = session_metrics.join(
    forms.select("session_id", "uid", "phq9_score", "phq9_severity", "bdi_score", "bdi_severity"),
    on="session_id",
    how="inner",
)

df = df_joined.toPandas()
print(f"Dataset: {len(df)} sessions, {df['uid'].nunique()} unique users")

### 1.2 Aggregate to user level

In [0]:
FEATURE_COLS = [c for c in df.columns if c.startswith(("avg_", "first_fix_prob_", "second_fix_prob_"))]
NUMERIC_COLS = FEATURE_COLS + ["phq9_score", "bdi_score"]

df = df.groupby("uid", as_index=False)[NUMERIC_COLS].mean()

print(f"After per-user aggregation: {len(df)} users")
print(f"PHQ-9: min={df['phq9_score'].min():.1f}, max={df['phq9_score'].max():.1f}, mean={df['phq9_score'].mean():.1f}")

## 2. Define targets and feature sets

### 2.1 PHQ-9 targets

Binary cutoff: PHQ-9 ≥ 10 (standard Kroenke clinical threshold).
Severity classes (Kroenke et al., 2001):
- 0 = minimal (0–4)
- 1 = mild (5–9)
- 2 = moderate (10–14)
- 3 = moderately severe (15–19)
- 4 = severe (20–27)

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)

def phq9_severity_from_score(s):
    if s <= 4:  return 0   # minimal
    if s <= 9:  return 1   # mild
    if s <= 14: return 2   # moderate
    if s <= 19: return 3   # moderately severe
    return 4               # severe

df["phq9_severity_class"] = df["phq9_score"].apply(phq9_severity_from_score)

print("PHQ-9")
print(f"  Binary (>=10): {df['phq9_depressed'].value_counts().to_dict()}")
print(f"  Multi-class:   {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

### 2.2 BDI-II targets

Binary cutoff: BDI-II ≥ 14 (Beck et al., 1996 threshold for mild or worse depression; 81% sensitivity, 92% specificity against DSM-IV MDD).
Severity classes (Beck et al., 1996):
- 0 = minimal (0–13)
- 1 = mild (14–19)
- 2 = moderate (20–28)
- 3 = severe (29–63)

In [0]:
df["bdi_depressed"] = (df["bdi_score"] >= 14).astype(int)

def bdi_severity_from_score(s):
    if s <= 13: return 0   # minimal
    if s <= 19: return 1   # mild
    if s <= 28: return 2   # moderate
    return 3               # severe

df["bdi_severity_class"] = df["bdi_score"].apply(bdi_severity_from_score)

print("BDI-II")
print(f"  Binary (>=14): {df['bdi_depressed'].value_counts().to_dict()}")
print(f"  Multi-class:   {df['bdi_severity_class'].value_counts().sort_index().to_dict()}")

### 2.3 Feature sets

In [0]:
ALL_FEATURES = [
    "avg_fixation_count", "avg_fixation_duration_ms", "avg_total_fixation_duration_ms",
    "avg_fixation_rate", "avg_fixation_bias",
    "avg_scanpath_length", "avg_saccade_count", "avg_saccade_rate", "avg_saccade_amplitude",
    "avg_blink_count", "avg_blink_rate",
    "avg_transition_density", "avg_gaze_entropy",
    "avg_first_fixation_duration_ms", "avg_second_fixation_duration_ms",
    "first_fix_prob_negative", "first_fix_prob_positive", "first_fix_prob_neutral",
    "second_fix_prob_negative", "second_fix_prob_positive", "second_fix_prob_neutral",
    "avg_dwell_time_negative", "avg_dwell_time_positive", "avg_dwell_time_neutral",
    "avg_dwell_500ms_negative", "avg_dwell_500ms_positive", "avg_dwell_500ms_neutral",
    "avg_fix_proportion_negative", "avg_fix_proportion_positive", "avg_fix_proportion_neutral",
    "avg_fix_count_negative", "avg_fix_count_positive", "avg_fix_count_neutral",
    "avg_revisit_count_negative", "avg_revisit_count_positive", "avg_revisit_count_neutral",
    "avg_ttff_negative", "avg_ttff_positive", "avg_ttff_neutral",
]

SIGNIFICANT_FEATURES = [
    "avg_fixation_bias",
    "avg_fix_proportion_positive",
    "avg_fix_count_positive",
    "avg_dwell_time_positive",
    "avg_blink_count",
    "avg_blink_rate",
    "avg_fixation_rate",
    "avg_saccade_rate",
    "avg_fixation_count",
    "avg_saccade_count",
    "avg_transition_density",
    "avg_saccade_amplitude",
    "second_fix_prob_neutral",
    "avg_ttff_negative",
]

THEORY_FEATURES = [
    "avg_fixation_bias",
    "avg_dwell_time_negative",
    "avg_dwell_time_positive",
    "avg_fix_proportion_negative",
    "avg_fix_proportion_positive",
    "first_fix_prob_negative",
    "first_fix_prob_positive",
    "avg_revisit_count_negative",
    "avg_revisit_count_positive",
    "avg_blink_rate",
    "avg_scanpath_length",
]

FEATURE_SETS = {
    "All Features": ALL_FEATURES,
    "Significant Only": SIGNIFICANT_FEATURES,
    "Theory-Driven": THEORY_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
target_cols = [
    "phq9_score", "phq9_depressed", "phq9_severity_class",
    "bdi_score",  "bdi_depressed",  "bdi_severity_class",
]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. Binary classification

Four binary analyses crossing two depression scales with two thresholding strategies:
- clinical cutoff (PHQ-9 ≥ 10, BDI-II ≥ 14)
- extremes task (drop the ambiguous middle; compare minimal vs severe)


### 4.1 PHQ-9 cutoff (≥ 10)

#### 4.1.1 Run classification

In [0]:
y_phq9_binary = df_clean["phq9_depressed"].values
phq9_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups)

#### 4.1.2 Results

In [0]:
print(phq9_binary_df.to_string(index=False))

#### 4.1.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups, phq9_binary_df)

### 4.2 PHQ-9 extremes (minimal vs severe)

Minimal (PHQ-9 ≤ 4) vs Severe (PHQ-9 ≥ 20). Users with intermediate scores are excluded.

#### 4.2.1 Run classification

In [0]:
df_phq9_extreme = df_clean[(df_clean["phq9_score"] <= 4) | (df_clean["phq9_score"] >= 20)].copy()
df_phq9_extreme["phq9_extreme"] = (df_phq9_extreme["phq9_score"] >= 20).astype(int)

groups_phq9_extreme = df_phq9_extreme["uid"].values

print(f"PHQ-9 extremes dataset: {len(df_phq9_extreme)} users")
print(f"  Minimal (<=4):  {(df_phq9_extreme['phq9_extreme'] == 0).sum()}")
print(f"  Severe  (>=20): {(df_phq9_extreme['phq9_extreme'] == 1).sum()}")

y_phq9_extreme = df_phq9_extreme["phq9_extreme"].values
phq9_extreme_df = run_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme)

#### 4.2.2 Results

In [0]:
print(phq9_extreme_df.to_string(index=False))

#### 4.2.3 Best model

In [0]:
plot_best_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme, phq9_extreme_df)

### 4.3 BDI-II cutoff (≥ 14)

#### 4.3.1 Run classification

In [0]:
y_bdi_binary = df_clean["bdi_depressed"].values
bdi_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups)

#### 4.3.2 Results

In [0]:
print(bdi_binary_df.to_string(index=False))

#### 4.3.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups, bdi_binary_df)

### 4.4 BDI-II extremes (minimal vs severe)

Minimal (BDI ≤ 13) vs Severe (BDI ≥ 29). Users with intermediate scores are excluded.

#### 4.4.1 Run classification

In [0]:
df_bdi_extreme = df_clean[(df_clean["bdi_score"] <= 13) | (df_clean["bdi_score"] >= 29)].copy()
df_bdi_extreme["bdi_extreme"] = (df_bdi_extreme["bdi_score"] >= 29).astype(int)

groups_bdi_extreme = df_bdi_extreme["uid"].values

print(f"BDI-II extremes dataset: {len(df_bdi_extreme)} users")
print(f"  Minimal (<=13): {(df_bdi_extreme['bdi_extreme'] == 0).sum()}")
print(f"  Severe  (>=29): {(df_bdi_extreme['bdi_extreme'] == 1).sum()}")

y_bdi_extreme = df_bdi_extreme["bdi_extreme"].values
bdi_extreme_df = run_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme)

#### 4.4.2 Results

In [0]:
print(bdi_extreme_df.to_string(index=False))

#### 4.4.3 Best model

In [0]:
plot_best_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme, bdi_extreme_df)

## 5. Multi-class classification

### 5.1 PHQ-9 (5 severity classes)

#### 5.1.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_phq9_multi = df_clean["phq9_severity_class"].values.astype(int)
phq9_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups)

#### 5.1.2 Results

In [0]:
print(phq9_multi_df.to_string(index=False))

#### 5.1.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups, phq9_multi_df, PHQ9_LABELS)

### 5.2 BDI-II (4 severity classes)

#### 5.2.1 Run classification

In [0]:
BDI_LABELS = ["Minimal", "Mild", "Moderate", "Severe"]
y_bdi_multi = df_clean["bdi_severity_class"].values.astype(int)
bdi_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups)

#### 5.2.2 Results

In [0]:
print(bdi_multi_df.to_string(index=False))

#### 5.2.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups, bdi_multi_df, BDI_LABELS)

## 6. Regression

### 6.1 PHQ-9 score prediction

#### 6.1.1 Run regression

In [0]:
y_phq9_reg = df_clean["phq9_score"].values
phq9_reg_df = run_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups)

#### 6.1.2 Results

In [0]:
print(phq9_reg_df.to_string(index=False))

#### 6.1.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups, phq9_reg_df, score_name="PHQ-9")

### 6.2 BDI-II score prediction

#### 6.2.1 Run regression

In [0]:
y_bdi_reg = df_clean["bdi_score"].values
bdi_reg_df = run_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups)

#### 6.2.2 Results

In [0]:
print(bdi_reg_df.to_string(index=False))

#### 6.2.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups, bdi_reg_df, score_name="BDI-II")

## 7. PHQ-9 vs BDI-II convergent-validity comparison

Side-by-side best performance across all tasks. If the two scales produce similar numbers, the gaze-based depression signal generalizes across questionnaires.

In [0]:
def _best_auc(df_res):
    return df_res.sort_values("auc_roc", ascending=False).iloc[0]

def _best_f1(df_res):
    return df_res.sort_values("f1_weighted", ascending=False).iloc[0]

def _best_r2(df_res):
    return df_res.sort_values("r2", ascending=False).iloc[0]

rows = []

p = _best_auc(phq9_binary_df)
b = _best_auc(bdi_binary_df)
rows.append({"task": "Binary cutoff",
             "phq9_metric": "AUC",  "phq9_value": p["auc_roc"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "AUC",   "bdi_value": b["auc_roc"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

p = _best_auc(phq9_extreme_df)
b = _best_auc(bdi_extreme_df)
rows.append({"task": "Binary extremes",
             "phq9_metric": "AUC",  "phq9_value": p["auc_roc"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "AUC",   "bdi_value": b["auc_roc"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

p = _best_f1(phq9_multi_df)
b = _best_f1(bdi_multi_df)
rows.append({"task": "Multi-class",
             "phq9_metric": "F1",   "phq9_value": p["f1_weighted"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "F1",    "bdi_value": b["f1_weighted"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

p = _best_r2(phq9_reg_df)
b = _best_r2(bdi_reg_df)
rows.append({"task": "Regression",
             "phq9_metric": "R2",   "phq9_value": p["r2"],
             "phq9_model": f"{p['model']} / {p['feature_set']}",
             "bdi_metric": "R2",    "bdi_value": b["r2"],
             "bdi_model": f"{b['model']} / {b['feature_set']}"})

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

## 8. Feature importance

Feature importance on the PHQ-9 binary task (Random Forest, all features). Presented once because PHQ-9 and BDI-II converge on similar predictors.

In [0]:
plot_feature_importance(df_clean, ALL_FEATURES, y_phq9_binary, title="Feature importance (PHQ-9 binary, all features)")

## 9. Summary

### 9.1 PHQ-9

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(phq9_binary_df, phq9_multi_df, phq9_reg_df, feature_order, title="PHQ-9 — static features")

### 9.2 BDI-II

In [0]:
plot_summary(bdi_binary_df, bdi_multi_df, bdi_reg_df, feature_order, title="BDI-II — static features")